# Part 2: Web-Search — Inverted Index and Search Engine

## Assignment 4 — Building an Inverted Index

We implement the following classes as specified:
- `MySet` — Set with union/intersection
- `Position` — Tuple (page, word_index)
- `WordEntry` — Word + list of positions
- `PageIndex` — Per-document word-entry map
- `PageEntry` — Reads file, builds page index
- `MyHashTable` — Hash-based inverted index storage
- `InvertedPageIndex` — Global inverted index across pages
- `SearchEngine` — Action dispatcher

In [12]:
import os
import re
import math

# Path to webpages folder
WEBPAGES_DIR = "Q2- webSearch/webpages"

# Connector words (stop words) — do NOT store, but DO count for word indices
CONNECTOR_WORDS = {"a", "an", "the", "they", "these", "this", "for", "is", "are",
                   "was", "of", "or", "and", "does", "will", "whose"}

# Punctuation to replace with space
PUNCTUATION = set('{}[]<>=().,;\'"?#!-:')

# Plural -> Singular mapping (exhaustive as per assignment)
PLURAL_TO_SINGULAR = {
    "stacks": "stack",
    "structures": "structure",
    "applications": "application",
}

In [ ]:
def normalize_word(word):
    #Convert to lowercase and map plural to singular.
    word = word.lower()
    if word in PLURAL_TO_SINGULAR:
        word = PLURAL_TO_SINGULAR[word]
    return word


def replace_punctuation(text):
    #Replace all punctuation characters with spaces.
    result = []
    for ch in text:
        if ch in PUNCTUATION:
            result.append(' ')
        else:
            result.append(ch)
    return ''.join(result)


def tokenize(text):
    #
    #Replace punctuation with spaces, split into words.
    #Returns list of (word_index, normalized_word) tuples.
    #Word index counts ALL words (including connectors) starting from 1.
    #Connector words are included in the index count but not stored.
    #
    text = replace_punctuation(text)
    raw_words = text.split()
    tokens = []
    for i, w in enumerate(raw_words):
        norm = normalize_word(w)
        # word_index is 1-based, counts all words including connectors
        tokens.append((i + 1, norm))
    return tokens

## Class Implementations

In [ ]:
class MySet:
    #A simple set data structure.

    def __init__(self):
        self._elements = []

    def addElement(self, element):
        #Add element to the set (no duplicates).
        if element not in self._elements:
            self._elements.append(element)

    def union(self, otherSet):
        #Return a new MySet representing the union.
        result = MySet()
        for e in self._elements:
            result.addElement(e)
        for e in otherSet._elements:
            result.addElement(e)
        return result

    def intersection(self, otherSet):
        #Return a new MySet representing the intersection.
        result = MySet()
        for e in self._elements:
            if e in otherSet._elements:
                result.addElement(e)
        return result

    def elements(self):
        return self._elements

    def __len__(self):
        return len(self._elements)

    def __contains__(self, item):
        return item in self._elements

    def __iter__(self):
        return iter(self._elements)

    def __repr__(self):
        return f"MySet({self._elements})"

In [ ]:
class Position:
    #Represents a tuple <page p, word position i>.

    def __init__(self, p, wordIndex):
        #p: PageEntry, wordIndex: int (1-based position in document).
        self._pageEntry = p
        self._wordIndex = wordIndex

    def getPageEntry(self):
        return self._pageEntry

    def getWordIndex(self):
        return self._wordIndex

    def __repr__(self):
        return f"({self._pageEntry.pageName}, {self._wordIndex})"

    def __eq__(self, other):
        if not isinstance(other, Position):
            return False
        return (self._pageEntry.pageName == other._pageEntry.pageName and
                self._wordIndex == other._wordIndex)

    def __hash__(self):
        return hash((self._pageEntry.pageName, self._wordIndex))

In [ ]:
class WordEntry:
    #Stores the list of word indices where a string is present in document(s).

    def __init__(self, word):
        self._word = word
        self._positions = []  # list of Position objects

    @property
    def word(self):
        return self._word

    def addPosition(self, position):
        #Add a single Position entry.
        self._positions.append(position)

    def addPositions(self, positions):
        #Add multiple Position entries.
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self):
        #Return a list of all Position entries.
        return self._positions

    def getTermFrequency(self, pageEntry):
        #Return the term frequency of the word in the given webpage.
        # Count occurrences of this word in the given page
        count = sum(1 for p in self._positions if p.getPageEntry().pageName == pageEntry.pageName)
        total_words = pageEntry.getTotalWords()
        if total_words == 0:
            return 0.0
        return count / total_words

    def __repr__(self):
        return f"WordEntry({self._word}, positions={self._positions})"

In [ ]:
class PageIndex:
    #Stores one word-entry for each unique word in a document.

    def __init__(self):
        self._word_entries = {}  # word -> WordEntry

    def addPositionForWord(self, word, position):
        #Add position to the word-entry for 'word'. Creates a new entry if needed.
        if word not in self._word_entries:
            self._word_entries[word] = WordEntry(word)
        self._word_entries[word].addPosition(position)

    def getWordEntries(self):
        #Return a list of all WordEntry objects.
        return list(self._word_entries.values())

    def getWordEntry(self, word):
        #Return the WordEntry for a given word, or None.
        return self._word_entries.get(word, None)

    def __repr__(self):
        return f"PageIndex({list(self._word_entries.keys())})"

In [ ]:
class PageEntry:
    #Stores information related to a webpage. Reads the file and creates the page index.

    def __init__(self, pageName):
        self.pageName = pageName
        self._pageIndex = PageIndex()
        self._totalWords = 0

        # Read the file and build the page index
        filepath = os.path.join(WEBPAGES_DIR, pageName)
        with open(filepath, 'r') as f:
            text = f.read()

        tokens = tokenize(text)
        self._totalWords = len(tokens)  # total words including connectors

        for word_idx, word in tokens:
            # Do NOT store connector words, but they count for word indices
            if word not in CONNECTOR_WORDS:
                pos = Position(self, word_idx)
                self._pageIndex.addPositionForWord(word, pos)

    def getPageIndex(self):
        #Return the PageIndex of this webpage.
        return self._pageIndex

    def getTotalWords(self):
        #Return the total word count (including connectors).
        return self._totalWords

    def __repr__(self):
        return f"PageEntry({self.pageName})"

    def __eq__(self, other):
        if not isinstance(other, PageEntry):
            return False
        return self.pageName == other.pageName

    def __hash__(self):
        return hash(self.pageName)

In [ ]:
class MyHashTable:
    # Hash table used by InvertedPageIndex. Maps a word to its WordEntry.

    def __init__(self, size=1024):
        self._size = size
        self._table = [None] * size

    def getHashIndex(self, word):
        #Hash function mapping a string to an index.
        return hash(word) % self._size

    def addPositionsForWord(self, wordEntry):
    #
    #    Add a WordEntry to the hash table.
    #    If no entry exists for this word, create one.
    #    If one exists, merge positions into it.
    #
        idx = self.getHashIndex(wordEntry.word)
        if self._table[idx] is None:
            # No entry at this index; create a new WordEntry
            new_entry = WordEntry(wordEntry.word)
            new_entry.addPositions(wordEntry.getAllPositionsForThisWord())
            self._table[idx] = [new_entry]
        else:
            # Check if word already exists in this bucket
            for existing in self._table[idx]:
                if existing.word == wordEntry.word:
                    existing.addPositions(wordEntry.getAllPositionsForThisWord())
                    return
            # Word not found in bucket; add new entry
            new_entry = WordEntry(wordEntry.word)
            new_entry.addPositions(wordEntry.getAllPositionsForThisWord())
            self._table[idx].append(new_entry)

    def getWordEntry(self, word):
        # Get the WordEntry for a given word, or None.
        idx = self.getHashIndex(word)
        if self._table[idx] is None:
            return None
        for entry in self._table[idx]:
            if entry.word == word:
                return entry
        return None

In [ ]:
class InvertedPageIndex:
    # Global inverted index across all added pages.

    def __init__(self):
        self._hashTable = MyHashTable()
        self._pages = {}  # pageName -> PageEntry

    def addPage(self, pageEntry):
        #Add a new PageEntry to the inverted page index.
        self._pages[pageEntry.pageName] = pageEntry
        # Merge all word entries from this page into the global inverted index
        for wordEntry in pageEntry.getPageIndex().getWordEntries():
            self._hashTable.addPositionsForWord(wordEntry)

    def getPagesWhichContainWord(self, word):
        #Return a MySet of PageEntry objects that contain the given word.
        word = normalize_word(word)
        result = MySet()
        entry = self._hashTable.getWordEntry(word)
        if entry is None:
            return result
        # Collect unique pages from all positions
        for pos in entry.getAllPositionsForThisWord():
            result.addElement(pos.getPageEntry())
        return result

    def getPositionsOfWordInPage(self, word, pageName):
        #Return list of word indices where 'word' appears in page 'pageName', or None.
        word = normalize_word(word)
        entry = self._hashTable.getWordEntry(word)
        if entry is None:
            return []
        positions = []
        for pos in entry.getAllPositionsForThisWord():
            if pos.getPageEntry().pageName == pageName:
                positions.append(pos.getWordIndex())
        return sorted(positions)

    def hasPage(self, pageName):
        #Check if a page has been added.
        return pageName in self._pages

In [ ]:
class SearchEngine:
    #Main search engine that processes actions and returns results.

    def __init__(self):
        self._invertedIndex = InvertedPageIndex()

    def performAction(self, action_line):
        #
        #Process one action line.
        #Returns a string result or None (for addPage which produces no output).
        #
        parts = action_line.strip().split()
        action = parts[0]

        if action == "addPage":
            pageName = parts[1]
            pageEntry = PageEntry(pageName)
            self._invertedIndex.addPage(pageEntry)
            return None

        elif action == "queryFindPagesWhichContainWord":
            word = parts[1]
            pages = self._invertedIndex.getPagesWhichContainWord(word)
            if len(pages) == 0:
                return f"No webpage contains word {word}"
            # Return page names sorted alphabetically, comma-separated
            return ", ".join(sorted(pe.pageName for pe in pages))

        elif action == "queryFindPositionsOfWordInAPage":
            word = parts[1]
            pageName = parts[2]
            # Check if page exists in the database first
            if not self._invertedIndex.hasPage(pageName):
                return f"No webpage {pageName} found"
            positions = self._invertedIndex.getPositionsOfWordInPage(word, pageName)
            if not positions:
                return f"Webpage {pageName} does not contain word {word}"
            # Return positions comma-separated as per spec
            return ", ".join(str(p) for p in positions)

        else:
            return f"Unknown action: {action}"

## Run Actions and Verify Against Expected Answers

In [22]:
# Read actions
actions_path = os.path.join(os.path.dirname(WEBPAGES_DIR), "actions.txt")
with open(actions_path, 'r') as f:
    actions = [line.strip() for line in f if line.strip()]

# Read expected answers
answers_path = os.path.join(os.path.dirname(WEBPAGES_DIR), "answers.txt")
with open(answers_path, 'r') as f:
    expected_answers = [line.strip() for line in f if line.strip()]

# Run the search engine
engine = SearchEngine()
actual_answers = []

for action in actions:
    result = engine.performAction(action)
    if result is not None:
        actual_answers.append(result)
        print(result)

print("\n--- Verification ---")
all_match = True
for i, (exp, act) in enumerate(zip(expected_answers, actual_answers)):
    match = "✓" if exp == act else "✗"
    if exp != act:
        all_match = False
        print(f"  {match} Line {i+1}:")
        print(f"    Expected: {exp}")
        print(f"    Actual:   {act}")
    else:
        print(f"  {match} Line {i+1}: {act}")

if len(expected_answers) != len(actual_answers):
    print(f"  WARNING: Expected {len(expected_answers)} answers, got {len(actual_answers)}")
    all_match = False

if all_match:
    print("\nAll answers match! ✓")

No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine

--- Verification ---
  ✓ Line 1: No webpage contains word delhi
  ✓ Line 2: stack_datastructure_wiki
  ✓ Line 3: stack_datastructure_wiki
  ✓ Line 4: Webpage stack_datastructure_wiki does not contain word magazines
  ✓ Line 5: No webpage contains word allain
  ✓ Line 6: stack_cprogramming
  ✓ Line 7: stack_cprogramming
  ✓ Line 8: stack_cprogramming
  ✓ Line 9: stack_oracle
  ✓ Line 10: stack_cprogramming, stack_datastructure_wiki, stackoverflow
  ✓ Line 11: stackmagazine

All answers match! ✓
